In [1]:
import requests
from lxml import etree
import zipfile
from io import BytesIO

INDEX_URL = "https://www.gesetze-im-internet.de/gii-toc.xml"

index_response = requests.get(INDEX_URL)
index_root = etree.fromstring(index_response.content)

In [2]:
first = index_root[0]

title = first.findtext("title")
link = first.findtext("link")

print(title)
print(link)

Gesetz über die Ausprägung einer 1-DM-Goldmünze und die Errichtung der Stiftung "Geld und Währung"
http://www.gesetze-im-internet.de/1-dm-goldm_nzg/xml.zip


In [3]:
law_response = requests.get(link)

z = zipfile.ZipFile(BytesIO(law_response.content))

xml_name = z.namelist()[0]

xml = z.read(xml_name)

law_root = etree.fromstring(xml)

In [4]:
items = []

for item in index_root:
    items.append(
        {
            "title": item.findtext("title"),
            "link": item.findtext("link"),
        }
    )

len(items)

6125

In [5]:
items[:3]

[{'title': 'Gesetz über die Ausprägung einer 1-DM-Goldmünze und die Errichtung der Stiftung "Geld und Währung"',
  'link': 'http://www.gesetze-im-internet.de/1-dm-goldm_nzg/xml.zip'},
 {'title': 'Erstes Gesetz zur Vereinheitlichung und Neuregelung des Besoldungsrechts in Bund und Ländern',
  'link': 'http://www.gesetze-im-internet.de/besvng_1/xml.zip'},
 {'title': 'Erste Verordnung zur Durchführung des Bundes-Immissionsschutzgesetzes *)',
  'link': 'http://www.gesetze-im-internet.de/bimschv_1_2010/xml.zip'}]

In [6]:
[x for x in items if "bürger" in (x["title"] or "").lower()][:5]

[{'title': 'Gesetz über Rechtsberatung und Vertretung für Bürger mit geringem Einkommen',
  'link': 'http://www.gesetze-im-internet.de/berathig/xml.zip'},
 {'title': 'Bürgerliches Gesetzbuch',
  'link': 'http://www.gesetze-im-internet.de/bgb/xml.zip'},
 {'title': 'Bekanntmachung betreffend Ausführungsbestimmungen zu den §§ 980, 981, 983 des Bürgerlichen Gesetzbuchs',
  'link': 'http://www.gesetze-im-internet.de/bgbabest_1898/xml.zip'},
 {'title': 'Einführungsgesetz zum Bürgerlichen Gesetzbuche',
  'link': 'http://www.gesetze-im-internet.de/bgbeg/xml.zip'},
 {'title': 'Gesetz zur Europäischen Bürgerinitiative',
  'link': 'http://www.gesetze-im-internet.de/ebig/xml.zip'}]

In [7]:
[x for x in items if "miet" in (x["title"] or "").lower()][:5]

[{'title': 'Gesetz über Altschuldenhilfen für Kommunale Wohnungsunternehmen, Wohnungsgenossenschaften und private Vermieter in dem in Artikel 3 des Einigungsvertrages genannten Gebiet',
  'link': 'http://www.gesetze-im-internet.de/altschg/xml.zip'},
 {'title': 'Verordnung über die gewerbsmäßige Vermietung von Sportbooten sowie deren Benutzung auf den Binnenschifffahrtsstraßen',
  'link': 'http://www.gesetze-im-internet.de/sportbootvermv-bin2000/xml.zip'},
 {'title': 'Gesetz zur Verlängerung der Regelung über die Anmietung von Kraftfahrzeugen im Werkverkehr nach dem Einigungsvertrag',
  'link': 'http://www.gesetze-im-internet.de/kfzanmverlg/xml.zip'},
 {'title': 'Verordnung zur Einstufung der Gemeinden in eine Mietniveaustufe im Sinne des § 254 des Bewertungsgesetzes',
  'link': 'http://www.gesetze-im-internet.de/mietneinv/xml.zip'},
 {'title': 'Gesetz über die Pfändung von Miet- und Pachtzinsforderungen wegen Ansprüche aus öffentlichen Grundstückslasten',
  'link': 'http://www.gesetze-

In [8]:
bgb = [x for x in items if x["title"] == "Bürgerliches Gesetzbuch"][0]

bgb["link"]

'http://www.gesetze-im-internet.de/bgb/xml.zip'

In [9]:
bgb_response = requests.get(bgb["link"])

z = zipfile.ZipFile(BytesIO(bgb_response.content))

z.namelist()

['BJNR001950896.xml']

In [10]:
bgb_xml = z.read("BJNR001950896.xml")
bgb_root = etree.fromstring(bgb_xml)

In [11]:
bgb_root.tag

'dokumente'

In [12]:
for child in bgb_root[0]:
    print(child.tag)

metadaten
textdaten


In [13]:
textdaten = bgb_root.find(".//textdaten")

for child in textdaten:
    print(child.tag)

text
fussnoten


In [14]:
text = bgb_root.find(".//textdaten/text")

print(len(text))

1


In [15]:
for child in text[:5]:
    print(child.tag)

Content


In [16]:
content = bgb_root.find(".//textdaten/text/Content")

print(len(content))

1


In [17]:
for child in content[:10]:
    print(child.tag)

P


In [18]:
from lxml import etree

print(etree.tostring(content[0], pretty_print=True, encoding="unicode")[:2000])

<P>
  <noindex>Dieses Gesetz dient der Umsetzung folgender Richtlinien: <DL Font="normal" Type="arabic"><DT>1.</DT><DD Font="normal"><LA Size="normal">Richtlinie 76/207/EWG des Rates vom 9. Februar 1976 zur Verwirklichung des Grundsatzes der Gleichbehandlung von Männern und Frauen hinsichtlich des Zugangs zur Beschäftigung, zur Berufsbildung und zum beruflichen Aufstieg sowie in Bezug auf die Arbeitsbedingungen (ABl. EG Nr. L 39 S. 40),</LA></DD><DT>2.</DT><DD Font="normal"><LA Size="normal">Richtlinie 77/187/EWG des Rates vom 14. Februar 1977 zur Angleichung der Rechtsvorschriften der Mitgliedstaaten über die Wahrung von Ansprüchen der Arbeitnehmer beim Übergang von Unternehmen, Betrieben oder Betriebsteilen (ABl. EG Nr. L 61 S. 26),</LA></DD><DT>3.</DT><DD Font="normal"><LA Size="normal">Richtlinie 85/577/EWG des Rates vom 20. Dezember 1985 betreffend den Verbraucherschutz im Falle von außerhalb von Geschäftsräumen geschlossenen Verträgen (ABl. EG Nr. L 372 S. 31),</LA></DD><DT>4.</D

In [19]:
from lxml import etree

text = etree.tostring(content, encoding="unicode")

print("§" in text)

False


In [20]:
for element in content.iter():
    if element.text and "Bürgerliches Gesetzbuch" in element.text:
        print(element.tag, element.text[:200])